# 5. Frozen text representations (GPU)

Every article must fit whole: token counts for every tokenizer, no article above the limit. Then the vectors of
Qwen3-Embedding-8B (instruction + article, last token) and Mistral-Small-24B Base (4 bit, mean of the tokens, layers 27
and 40; the linear models use 27).

In [1]:
QWEN = "Qwen/Qwen3-Embedding-8B"
MISTRAL = "mistralai/Mistral-Small-24B-Base-2501"
BERT = "answerdotai/ModernBERT-base"
QWEN_BUDGET = 32768           # tokens per batch
MISTRAL_BUDGET = 8192         # smaller: all hidden states stay in memory
GPU_GB = 22

In [2]:
# Parameters
GPU_GB = 19
QWEN_BUDGET = 16384


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, _ = MO.load()
texts = art.text.fillna("").tolist()
C.EMBEDDINGS.mkdir(parents=True, exist_ok=True)

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


In [4]:
rows = []
for name, model, prefix in [("ModernBERT", BERT, ""), ("Qwen3-8B", QWEN, f"Instruct: {MO.QWEN_INSTRUCTION}\nQuery:"), ("Mistral-24B", MISTRAL, "")]:
    n = MO.count_tokens(texts, model, prefix)
    limit = MO.MAX_TOKENS + (64 if prefix else 0)
    rows.append({"model": name, "median": int(np.median(n)), "99%": int(np.percentile(n, 99)), "max": int(n.max()),
                 "limit": limit, "truncated": int((n > limit).sum())})
tokens = pd.DataFrame(rows).set_index("model"); tokens.to_csv(C.OUT / "token_lengths.csv")
assert tokens.truncated.sum() == 0
tokens

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


,median,99%,max,limit,truncated
model,,,,,
ModernBERT,767,1930,2620,4096,0
Qwen3-8B,764,1904,2990,4160,0
Mistral-24B,758,1919,2980,4096,0


In [5]:
f = C.EMBEDDINGS / MO.EMBEDDING_FILES["Qwen3-8B"]
if not f.exists():
    MO.wait_for_gpu(GPU_GB, LOG)
    np.save(f, MO.qwen_embeddings(texts, QWEN, QWEN_BUDGET, LOG))
f = C.EMBEDDINGS / MO.EMBEDDING_FILES["Mistral-24B"]
if not f.exists():
    MO.wait_for_gpu(GPU_GB, LOG)
    E = MO.mistral_embeddings(texts, MISTRAL, [27, 40], MISTRAL_BUDGET, LOG, C.EMBEDDINGS / "mistral_partial.npy")
    np.save(C.EMBEDDINGS / "mistral_small_24b_layer40.npy", E[40]); np.save(f, E[27])
    (C.EMBEDDINGS / "mistral_partial.npy").unlink(missing_ok=True)
art[["article_id"]].to_csv(C.EMBEDDINGS / "article_order.csv", index=False)
for name, file in MO.EMBEDDING_FILES.items():
    E = np.load(C.EMBEDDINGS / file); print(name, E.shape, "finite:", np.isfinite(E).all())

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:  25%|██▌       | 1/4 [00:01<00:05,  1.93s/it]

Loading checkpoint shards:  50%|█████     | 2/4 [00:03<00:03,  1.95s/it]

Loading checkpoint shards:  75%|███████▌  | 3/4 [00:05<00:01,  1.97s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.25s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.51s/it]

... (56 outputs left out)

2026-11-11 22:55:13  Mistral: 24,850/25,524 articles


2026-11-11 23:06:24  Mistral: 25,250/25,524 articles


Qwen3-8B (25524, 4096) finite: True


Mistral-24B (25524, 5120) finite: True
